# Retrieval Test: Embedding Search vs Keyword Search

Match HYS feedback chunks against procedure articles.

**Goal:** find out whether semantic embedding search or simple keyword search better retrieves relevant lobbying statements for a given article.

**Swap the model** by changing `MODEL_NAME` (and optionally `QUERY_PREFIX` / `PASSAGE_PREFIX`) in the Config cell.

## Config — swap model here

In [60]:
# ── Model ────────────────────────────────────────────────────────────────────
# Swap MODEL_NAME to test a different transformer.
# Prefixes: some models need a short prefix on queries vs passages for retrieval.
#   Leave as "" if the model doesn't use them (e.g. bge-base, all-mpnet).
#   Use "query: " / "passage: " for E5 models.
#   Use "Represent this sentence for searching relevant passages: " / "" for bge-large instruct.

# MODEL_NAME      = "BAAI/bge-base-en-v1.5"   # 768-dim — same as legislation_embeddings
# QUERY_PREFIX    = ""                          # prepended to article text before encoding
# PASSAGE_PREFIX  = ""                          # prepended to chunk text before encoding

# Other models to try:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"  
QUERY_PREFIX = "" ; PASSAGE_PREFIX = ""
# MODEL_NAME = "intfloat/multilingual-e5-base"            ; QUERY_PREFIX = "query: " ; PASSAGE_PREFIX = "passage: "
# MODEL_NAME = "intfloat/multilingual-e5-large"           ; QUERY_PREFIX = "query: " ; PASSAGE_PREFIX = "passage: "
# MODEL_NAME = "BAAI/bge-large-en-v1.5"                   ; QUERY_PREFIX = "" ; PASSAGE_PREFIX = ""

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K = 5   # how many results to show per method

## 1. Imports & Supabase connection

In [63]:
import os
import re
import numpy as np
from dotenv import load_dotenv
from supabase import create_client
from sentence_transformers import SentenceTransformer

load_dotenv()
client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_ROLE_KEY"))
print("Connected to Supabase")

Connected to Supabase


## 2. Find procedures that exist in both HYS chunks and procedure_documents

In [64]:
# Supabase client paginates at 1000 rows by default — fetch all pages for both tables

def fetch_distinct_procedure_ids(table: str, extra_filter=None) -> set[str]:
    ids = set()
    offset = 0
    PAGE = 1000
    while True:
        q = client.table(table).select("procedure_id").range(offset, offset + PAGE - 1)
        if extra_filter:
            q = extra_filter(q)
        rows = q.execute().data or []
        ids.update(r["procedure_id"] for r in rows)
        if len(rows) < PAGE:
            break
        offset += PAGE
    return ids

chunk_procs = fetch_distinct_procedure_ids("hys_feedback_chunks")
doc_procs   = fetch_distinct_procedure_ids(
    "procedure_documents",
    extra_filter=lambda q: q.not_.is_("content_text", "null")
)

overlap = sorted(chunk_procs & doc_procs)
print(f"Procedures with HYS chunks:    {len(chunk_procs)}")
print(f"Procedures with document text: {len(doc_procs)}")
print(f"Overlap:                       {len(overlap)}\n")

for pid in overlap:
    n_chunks = (
        client.table("hys_feedback_chunks")
        .select("id", count="exact")
        .eq("procedure_id", pid)
        .execute()
        .count
    )
    doc_rows = (
        client.table("procedure_documents")
        .select("document_type")
        .eq("procedure_id", pid)
        .not_.is_("content_text", "null")
        .execute()
        .data or []
    )
    doc_types = [r["document_type"] for r in doc_rows]
    print(f"  {pid}  —  {n_chunks} chunks,  docs: {doc_types}")

Procedures with HYS chunks:    18
Procedures with document text: 740
Overlap:                       18

  2024/0061(COD)  —  39 chunks,  docs: ['committee_report', 'text_adopted', 'draft_report', 'opinion', 'opinion', 'opinion', 'opinion', 'commission_proposal']
  2024/0068(COD)  —  70 chunks,  docs: ['committee_report', 'draft_report', 'opinion', 'commission_proposal']
  2024/0318(COD)  —  152 chunks,  docs: ['committee_report', 'text_adopted', 'draft_report', 'opinion', 'commission_proposal']
  2024/0319(COD)  —  523 chunks,  docs: ['committee_report', 'text_adopted', 'draft_report', 'commission_proposal']
  2025/0071(COD)  —  163 chunks,  docs: ['commission_proposal', 'committee_report', 'text_adopted', 'draft_report']
  2025/0097(COD)  —  164 chunks,  docs: ['commission_proposal', 'draft_report']
  2025/0102(COD)  —  1735 chunks,  docs: ['commission_proposal', 'committee_report', 'text_adopted', 'draft_report', 'opinion', 'opinion', 'opinion']
  2025/0221(COD)  —  4240 chunks,  doc

## 3. Pick a procedure, fetch document text & split into articles

In [47]:
import re

# ── Pick procedure & document type ─────────────────────────────────────────────
PROCEDURE_ID = overlap[6]
DOC_TYPE     = "commission_proposal"

print(f"Procedure: {PROCEDURE_ID}")

doc_row = (
    client.table("procedure_documents")
    .select("document_id, content_text")
    .eq("procedure_id", PROCEDURE_ID)
    .eq("document_type", DOC_TYPE)
    .not_.is_("content_text", "null")
    .limit(1)
    .execute()
    .data or []
)
if not doc_row:
    raise ValueError(f"No {DOC_TYPE} with text for {PROCEDURE_ID}")

doc_id   = doc_row[0]["document_id"]
doc_text = doc_row[0]["content_text"]
print(f"Document:  {doc_id}  ({len(doc_text):,} chars)")

# ── Proposal date (used to filter pre-proposal feedback) ──────────────────────
proc_row = (
    client.table("procedures")
    .select("proposal_date")
    .eq("id", PROCEDURE_ID)
    .single()
    .execute()
    .data
)
proposal_date = proc_row["proposal_date"] or "2021-07-14"  # manual fallback
print(f"Proposal date: {proposal_date}")


# ── Splitter ───────────────────────────────────────────────────────────────────
def split_into_articles(text: str) -> list[dict]:
    """
    Split PDF-extracted legislative text into recitals and articles.

    Recitals: numbered (1), (2) ... between 'Whereas' and 'HAS/HAVE ADOPTED'
    Articles: lines that start with 'Article N' where N is a small integer (≤ 200),
              NOT mid-sentence treaty references like 'Article 114 TFEU'.
    """
    elements = []

    # ── Recitals ──────────────────────────────────────────────────────────────
    whereas_match = re.search(
        r'(?:Whereas|WHEREAS)[:\s]+(.*?)(?=HAS ADOPTED|HAVE ADOPTED|HEREBY ADOPTS)',
        text, re.DOTALL | re.IGNORECASE,
    )
    if whereas_match:
        block = whereas_match.group(1)
        hits  = list(re.finditer(r'(?:^|\n)\s*\((\d+)\)\s+', block))
        for i, m in enumerate(hits):
            num     = int(m.group(1))
            end     = hits[i + 1].start() if i + 1 < len(hits) else len(block)
            content = re.sub(r'\s+', ' ', block[m.end():end]).strip()
            if len(content) > 40:
                elements.append({
                    "element_type":   "recital",
                    "element_number": str(num),
                    "title":          None,
                    "content":        content,
                })

    # ── Articles ──────────────────────────────────────────────────────────────
    # Only match 'Article N' at the START of a line (after newline / whitespace),
    # followed by end-of-line or a short title — not mid-sentence treaty refs.
    art_pattern = re.compile(
        r'(?:^|\n)[ \t]*Article[ \t]+(\d+)[ \t]*([^\n]{0,80})\n(.*?)(?=(?:^|\n)[ \t]*Article[ \t]+\d+|\Z)',
        re.DOTALL | re.MULTILINE,
    )
    for m in art_pattern.finditer(text):
        num   = int(m.group(1))
        title = m.group(2).strip()
        body  = re.sub(r'\s+', ' ', m.group(3)).strip()

        # Skip treaty article references (large numbers, or body is almost empty)
        if num > 200:
            continue
        if len(body) < 40:
            continue
        # Skip if the "title" line contains typical treaty reference words
        if re.search(r'\bTFEU\b|\bTEU\b|\bCharter\b', title, re.IGNORECASE):
            continue

        elements.append({
            "element_type":   "article",
            "element_number": str(num),
            "title":          title if title else None,
            "content":        body[:5000],
        })

    return elements


articles_raw = split_into_articles(doc_text)
recitals_n   = sum(1 for a in articles_raw if a["element_type"] == "recital")
arts_n       = sum(1 for a in articles_raw if a["element_type"] == "article")
print(f"\nSplit:  {recitals_n} recitals,  {arts_n} articles")

print("\nFirst 15 elements:")
for a in articles_raw[:15]:
    title = a.get("title") or ""
    print(f"  [{a['element_type']:8s}] {a['element_number']:>4}  {title[:40]}  —  {a['content'][:80]}...")

Procedure: 2021/0218(COD)
Document:  COM(2021)0557  (200,349 chars)
Proposal date: 2021-07-14

Split:  47 recitals,  40 articles

First 15 elements:
  [recital ]    1    —  The European Green Deal5 establishes the objective of the Union becoming climate...
  [recital ]    2    —  Renewable energy plays a fundamental role in delivering the European Green Deal ...
  [recital ]    3    —  Directive (EU) 2018/2001 of the European Parliament and of the Council9 sets a b...
  [recital ]    4    —  There is a growing recognition of the need for alignment of bioenergy policies w...
  [recital ]    5    —  The rapid growth and increasing cost-competitiveness of renewable electricity pr...
  [recital ]    6    —  When calculating the share of renewables in a Member State, renewable fuels of n...
  [recital ]    7    —  Member States’ cooperation to promote renewable energy can take the form of stat...
  [recital ]    8    —  The Offshore Renewable Energy Strategy introduces an ambitious objectiv

## 4. Fetch HYS chunks & load model

In [48]:
# ── Fetch HYS chunks ──────────────────────────────────────────────────────────
PAGE = 1000
chunks_raw = []
offset = 0
if proposal_date:
    print(f"Filtering to pre-proposal feedback (before {proposal_date})")
else:
    print("Warning: proposal_date not set for this procedure — fetching all feedback")

while True:
    q = (
        client.table("hys_feedback_chunks")
        .select("id, chunk_index, chunk_text, organisation_name, transparency_reg_id, date_feedback")
        .eq("procedure_id", PROCEDURE_ID)
    )
    if proposal_date:
        q = q.lt("date_feedback", proposal_date)
    page = q.range(offset, offset + PAGE - 1).execute().data or []
    chunks_raw.extend(page)
    if len(page) < PAGE:
        break
    offset += PAGE
print(f"Chunks fetched: {len(chunks_raw)}  (pre-proposal: before {proposal_date})")

# ── Load model ────────────────────────────────────────────────────────────────
print(f"\nLoading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
DIM = model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {DIM}")

def embed(texts: list[str], prefix: str = "") -> np.ndarray:
    if prefix:
        texts = [prefix + t for t in texts]
    return model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

# ── Embed articles ────────────────────────────────────────────────────────────
article_texts = []
for a in articles_raw:
    header = f"{a['element_type']} {a['element_number']}"
    if a.get("title"):
        header += f": {a['title']}"
    article_texts.append(f"{header}\n{a['content']}".strip())

print(f"\nEmbedding {len(article_texts)} articles...")
article_emb = embed(article_texts, prefix=QUERY_PREFIX)
print(f"Article embeddings: {article_emb.shape}")

# ── Embed chunks ───────────────────────────────────────────────────────────────
chunk_texts = [c["chunk_text"] for c in chunks_raw]
print(f"\nEmbedding {len(chunk_texts)} chunks...")
chunk_emb = embed(chunk_texts, prefix=PASSAGE_PREFIX)
print(f"Chunk embeddings:   {chunk_emb.shape}")

Filtering to pre-proposal feedback (before 2021-07-14)
Chunks fetched: 3267  (pre-proposal: before 2021-07-14)

Loading sentence-transformers/all-mpnet-base-v2 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20606.51it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768

Embedding 87 articles...


Batches: 100%|██████████| 3/3 [00:04<00:00,  1.38s/it]


Article embeddings: (87, 768)

Embedding 3267 chunks...


Batches: 100%|██████████| 103/103 [02:17<00:00,  1.34s/it]

Chunk embeddings:   (3267, 768)


## 5. Retrieval functions + reciprocal matching function

In [57]:
def embedding_search(article_idx: int, top_k: int = TOP_K) -> list[dict]:
    query_vec = article_emb[article_idx]
    scores = chunk_emb @ query_vec
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for rank, i in enumerate(top_indices, 1):
        c = chunks_raw[i]
        results.append({
            "rank": rank,
            "score": float(scores[i]),
            "org": c.get("organisation_name") or "—",
            "date": str(c.get("date_feedback") or "")[:10],
            "text": c["chunk_text"],
        })
    return results


def keyword_search(article_idx: int, top_k: int = TOP_K) -> list[dict]:
    STOPWORDS = {
        "the", "a", "an", "of", "to", "in", "and", "or", "for", "is", "are",
        "be", "this", "that", "which", "with", "on", "at", "by", "from",
        "as", "its", "it", "shall", "may", "not", "any", "all", "such",
        "where", "when", "been", "have", "has", "their", "they", "than",
        "under", "within", "into", "also", "each", "these", "those",
    }
    article_text = article_texts[article_idx].lower()
    words = set(re.findall(r"\b[a-z]{4,}\b", article_text)) - STOPWORDS

    scored = [(sum(1 for w in words if w in c["chunk_text"].lower()), i)
              for i, c in enumerate(chunks_raw)]
    scored.sort(reverse=True)

    results = []
    for rank, (hits, i) in enumerate(scored[:top_k], 1):
        c = chunks_raw[i]
        results.append({
            "rank": rank,
            "score": hits,
            "org": c.get("organisation_name") or "—",
            "date": str(c.get("date_feedback") or "")[:10],
            "text": c["chunk_text"],
        })
    return results


def show_article(article_idx: int) -> None:
    a = articles_raw[article_idx]
    print(f"{'='*80}")
    print(f"ARTICLE  [{a['element_type']}] {a['element_number']}  —  {a.get('title') or ''}")
    print(f"version: {a.get('document_version') or 'unknown'}")
    print(f"{'-'*80}")
    print(a.get("content") or "")
    print(f"{'='*80}\n")


def show_results(results: list[dict], label: str, text_limit: int = 2000) -> None:
    print(f"\n{'─'*80}")
    print(f"  {label}")
    print(f"{'─'*80}")
    for r in results:
        score_str = f"{r['score']:.3f}" if isinstance(r['score'], float) else str(r['score'])
        print(f"\n  #{r['rank']}  score={score_str}  org={r['org']}  date={r['date']}")
        print(f"  {r['text'][:text_limit]}")
        if len(r['text']) > text_limit:
            print(f"  ... [{len(r['text']) - text_limit} more chars]")


def reciprocal_search(article_idx: int, top_k: int = TOP_K) -> list[dict]:
    """
    Returns top-k chunks by article->chunk similarity, annotated with whether
    the match is reciprocal: the chunk's best-matching article is this article.
    """
    # article -> chunks
    query_vec   = article_emb[article_idx]
    a2c_scores  = chunk_emb @ query_vec
    top_indices = np.argsort(a2c_scores)[::-1][:top_k]

    results = []
    for rank, ci in enumerate(top_indices, 1):
        # chunk -> articles (reverse direction)
        c2a_scores = article_emb @ chunk_emb[ci]
        best_art   = int(np.argmax(c2a_scores))
        reciprocal = (best_art == article_idx)

        c = chunks_raw[ci]
        results.append({
            "rank":         rank,
            "score":        float(a2c_scores[ci]),
            "reciprocal":   reciprocal,
            "best_article": best_art,
            "org":          c.get("organisation_name") or "—",
            "date":         str(c.get("date_feedback") or "")[:10],
            "text":         c["chunk_text"],
        })
    return results


def show_reciprocal_results(results: list[dict], text_limit: int = 300) -> None:
    reciprocal_n = sum(1 for r in results if r["reciprocal"])
    print(f"\n{'─'*80}")
    print(f"  RECIPROCAL MATCHING  ({reciprocal_n}/{len(results)} reciprocal)")
    print(f"{'─'*80}")
    for r in results:
        flag     = "RECIPROCAL" if r["reciprocal"] else "no"
        best_art = articles_raw[r["best_article"]]
        best_lbl = f"{best_art['element_type']} {best_art['element_number']}"
        print(f"\n  #{r['rank']}  score={r['score']:.3f}  [{flag}]  chunk_best_match={best_lbl}")
        print(f"  org={r['org']}  date={r['date']}")
        print(f"  {r['text'][:text_limit]}")
        if len(r["text"]) > text_limit:
            print(f"  ... [{len(r['text']) - text_limit} more chars]")


print("Retrieval functions ready.")

Retrieval functions ready.


## 6. Keyword vs. Embedding comparison

Set `ARTICLE_IDX` to whichever article you want to query against. Re-run this cell with different indices to explore.

In [ ]:
ARTICLE_IDX = 10  # <- change this to try different articles

show_article(ARTICLE_IDX)

emb_results = embedding_search(ARTICLE_IDX)
kw_results  = keyword_search(ARTICLE_IDX)

show_results(emb_results, f"EMBEDDING SEARCH  (model: {MODEL_NAME})")
show_results(kw_results,  "KEYWORD SEARCH    (word-hit count)")



ARTICLE  [recital] 11  —  
version: unknown
--------------------------------------------------------------------------------
Buildings have a large untapped potential to contribute effectively to the reduction in greenhouse gas emissions in the Union. The decarbonisation of heating and cooling in this sector through an increased share in production and use of renewable energy will be needed to meet the ambition set in the Climate Target Plan to achieve the Union objective of climate neutrality. However, progress on the use of renewables for heating and cooling has been stagnant in 15 Regulation (EU) 2018/1999 of the European Parliament and of the Council of 11 December 2018 on the Governance of the Energy Union and Climate Action, amending Regulations (EC) No 663/2009 and (EC) No 715/2009 of the European Parliament and of the Council, Directives 94/22/EC, 98/70/EC, 2009/31/EC, 2009/73/EC, 2010/31/EU, 2012/27/EU and 2013/30/EU of the European Parliament and of the Council, Council Direc

## Embedding Reciprocal Matching result

In [67]:
rec_results = reciprocal_search(ARTICLE_IDX)
show_reciprocal_results(rec_results)



────────────────────────────────────────────────────────────────────────────────
  RECIPROCAL MATCHING  (3/5 reciprocal)
────────────────────────────────────────────────────────────────────────────────

  #1  score=0.859  [no]  chunk_best_match=recital 23
  org=Veolia  date=2020-09-21
  In order to achieve this objective, sectoral targets would also have to be increased, in particular in the heating and cooling sector. This necessity is identified in the roadmap and is confirmed by the fact that as the recent BioHeat report shows, almost 80% of European heat still comes from fossil
  ... [1191 more chars]

  #2  score=0.856  [RECIPROCAL]  chunk_best_match=recital 11
  org=SolarPower Europe  date=2020-09-21
  In the NECPs, Belgium, Luxembourg and Spain introduced targets for the deployment of rooftop PV on their public building stock. Article 23 - Mainstreaming renewable energy in heating and cooling The heating and cooling sector represents almost half of the EU energy consumption, an

## 7. Overlap — keyword vs embeddings

Chunks that appear in both top-K lists are the strongest signal.

In [68]:
emb_texts = {r["text"] for r in emb_results}
kw_texts  = {r["text"] for r in kw_results}
shared    = emb_texts & kw_texts

print(f"Embedding-only results: {len(emb_texts - kw_texts)}")
print(f"Keyword-only results:   {len(kw_texts - emb_texts)}")
print(f"Shared by both:         {len(shared)}\n")

if shared:
    print("── Shared results ──")
    for t in shared:
        org = next((r["org"] for r in emb_results if r["text"] == t), "—")
        print(f"  org: {org}")
        print(f"  {t[:300]}\n")

Embedding-only results: 5
Keyword-only results:   5
Shared by both:         0

